# Dose/Exposure Data Pipeline

---
## Overview
Notebook 3 of 8.

This notebook completes the Bronze layer ingestion by loading dosing/exposure data.

Run the entire notebook to load the data.

---

## Source Data

- **File:** `dose.csv`
- **Location:** A Public GitHub repository set up especially for this exercise
- **Note:** Uses HTTPS URL so the notebook runs across any Databricks workspace

---

## Processing Steps

**1. Environment Setup**  
Ensures the required catalog and schemas exist for the Bronze layer

**2. Data Ingestion**  
Loads dose administration records into a DataFrame  
*Note: Uses Pandas to read from GitHub HTTPS URL*

**3. Provenance Metadata**  
Adds tracking metadata:
- Ingestion timestamp
- Unique ingest ID  
- Source filename

---

## Target

Data written to **bronze.dose_raw** Delta table using Spark MERGE for incremental updates.

---

In [0]:
from pyspark.sql.functions import lit, current_timestamp
from delta.tables import DeltaTable
from datetime import datetime
import pandas as pd

In [0]:
#global variables
#do not change the github link - this is a public repo from where the source data can be reliably sourced into any databricks workspace
CATALOG = "cluepoints_db_demo"
SCHEMA = "bronze"
GITHUB_URL = "https://raw.githubusercontent.com/prolfdog/cp_db_demo/main/"
ingestion_id = datetime.now().strftime('%Y%m%d_%H%M%S')
loaded_at = current_timestamp()
table_name = "dose"
full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

DataFrame[]

In [0]:
#use pandas to import from github as spark can't read https urls - NOT FOR PRODUCTION, ONLY FOR DEMO
#in production use spark.read.csv
#drop unnamed column on import
df_demog = spark.createDataFrame(
    pd.read_csv(f"{GITHUB_URL}dose.csv").drop(columns=['Unnamed: 0'])
)

In [0]:
#add data provenance variables to the data frame
df_demog = (df_demog 
    .withColumn("source_file", lit("dose.csv"))
    .withColumn("loaded_at", lit(loaded_at))
    .withColumn("ingestion_id", lit(ingestion_id))
)

In [0]:
if not spark.catalog.tableExists(full_table_name):
    #create the table if it doesnt exist
    df_demog.write.format("delta").saveAsTable(full_table_name)
else:
    #upsert data on unique columns
    #from analysis it seems the unique columns are subjid, drugname and dostrda even though drugname is NA in many cases
    target = DeltaTable.forName(spark, full_table_name)
    target.alias("target").merge(df_demog.alias("source"),
    "target.subjid = source.subjid AND target.drugname = source.drugname AND target.dostrda = source.dostrda").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
#display the table
display(spark.sql(f"select * from {full_table_name} limit 10"))

subjid,vnum,vname,dosinjec,dostrda_str,dostrda,drugname,satcnl1,satcnl3,center,repeat,drop,source_file,loaded_at,ingestion_id
208109018,10.0,DAY10,0.0,2004-03-09,2004-03-09,null,null,null,109,1,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,null,2005-06-11,2005-06-11,null,null,null,109,1,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,0.0,2005-03-05,2005-03-05,null,null,null,109,1,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,null,2005-11-26,2005-11-26,null,null,null,109,0,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,0.0,2004-02-19,2004-02-19,null,null,null,109,0,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,null,2005-02-15,2005-02-15,null,null,null,109,0,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,null,2005-09-27,2005-09-27,null,null,null,109,1,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,10.0,DAY10,0.0,2005-12-17,2005-12-17,null,null,null,109,0,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,0.0,2005-03-14,2005-03-14,null,null,null,109,1,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
208109018,9.0,DAY9,0.0,2005-06-25,2005-06-25,null,null,null,109,1,dunno,dose.csv,2026-01-05T08:29:14.335Z,20260105_082911
